# Ensemble: Person B's models + Faster R-CNN / RetinaNet + temporal rescoring

Three versions, each scored on the labelled val frames with the shared evaluator (native resolution, as in `09_final_pipeline.ipynb`):

| Version | Members | Question |
|---|---|---|
| **A** | Person B's final ensemble: 7 YOLO / RT-DETR runs, full frame + flip each, WBF IoU 0.65 (`runs/pipeline/final_config.json`, val mAP 0.2977 in 09) | Reference, reproduced on this machine |
| **B** | A + Faster R-CNN (multi-scale) and/or RetinaNet (multi-scale); WBF IoU re-tuned | Do the two-stage / anchor-based detectors add something the Ultralytics models miss? |
| **C** | the best of A / B + temporal rescoring (`src/temporal.py`) on the fused boxes | Does linking detections across frames still help after fusion? |

Temporal rescoring needs **every** frame of a clip (not only the labelled ones), so each member predicts all 2846 val frames and all 3811 test frames once; the boxes are cached in `runs/ensemble/` (resumable). Scores are always computed on the 359 labelled val frames only.

**Time on the RX 6800 XT (estimate):** about 1.5 h of GPU predictions (section 1 and section 5), then about 1 h of CPU for fusion. Nothing else may use the GPU meanwhile.
Set `FIXTURE=1` in the environment for a quick CPU check on a few frames.

**Outputs:** `runs/ensemble/results.csv` (all scored configurations) and three test submissions in `runs/ensemble/submissions/`.

## Setup

In [4]:
import json
import os
import pickle
import sys
import time
from pathlib import Path
from types import SimpleNamespace

import cv2
import numpy as np
import pandas as pd
import torch

SRC = Path.cwd().parent            # this notebook lives in src/experiments/
ROOT = SRC.parent
sys.path.insert(0, str(SRC))
from ultralytics import RTDETR, YOLO

from common.evaluate import evaluate_detections
from common.frcnn import load_faster_rcnn_run
from common.pipeline import combine, fuse, predict_passes, to_submission_rows
from common.retinanet import load_retinanet_run
from common.submission import write_submission
from infer import predict_image as tv_predict_image
from temporal import link_tracks, rescore

FIXTURE = os.environ.get("FIXTURE") == "1"
DATA_ROOT = ROOT / "multiple-object-video-detection-hw-2-ucu-ai-2026"
GT_FILE = ROOT / "data_processed" / "val" / "gt_eval_coco.json"
TEST_CSV = DATA_ROOT / "test_images.csv"
RUNS = ROOT / "runs"
OUT = RUNS / ("ensemble_fixture" if FIXTURE else "ensemble")
(OUT / "submissions").mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu", "| fixture:", FIXTURE)

device: AMD Radeon RX 6800 XT | fixture: False


## Members and frames

The 7 Ultralytics runs and their settings are the ones in Person B's `final_config.json` (each: full frame + horizontal flip, merged with class-wise NMS).
Faster R-CNN and RetinaNet use multi-scale inference (training scale 720/1280 + 1080/1920, merged with NMS), which helped Faster R-CNN on val and on the leaderboard.

In [5]:
MS = [(720, 1280), (1080, 1920)]
MEMBERS = {   # name: (family, run folder, input size)
    "yolo_low_scale_960":    ("yolo", RUNS / "yolo/low_scale_960", 960),
    "yolo_baseline_1280":    ("yolo", RUNS / "yolo/baseline", 1280),
    "yolo_final_960":        ("yolo", RUNS / "yolo/final_960", 960),
    "yolo_final_1280":       ("yolo", RUNS / "yolo/final_1280", 1280),
    "rtdetr_960":            ("rtdetr", RUNS / "rtdetr/imgsz_960", 960),
    "rtdetr_final_960":      ("rtdetr", RUNS / "rtdetr/final_960", 960),
    "rtdetr_final_960_s025": ("rtdetr", RUNS / "rtdetr/final_960_s025", 960),
    "frcnn_baseline_ms":     ("faster_rcnn", RUNS / "faster_rcnn/baseline", MS),
    "retinanet_baseline_ms": ("retinanet", RUNS / "retinanet/baseline", MS),
    # trained overnight (experiments/train_overnight.ipynb): 1280 px, original + test-like (shrunk, blurred) frames
    "yolo26m_1280_testres":  ("yolo", RUNS / "overnight/yolo26m_1280_testres", 1280),
    "rtdetr_l_1280_testres": ("rtdetr", RUNS / "overnight/rtdetr_l_1280_testres", 1280),
}
final_b = json.loads((RUNS / "pipeline" / "final_config.json").read_text())
B_MEMBERS = list(final_b["members"])
B_WBF_IOU = final_b["wbf_iou"]
assert set(B_MEMBERS) <= set(MEMBERS), set(B_MEMBERS) - set(MEMBERS)
assert all(tuple(v["passes"]) == ("full", "flip") and not v.get("tile_max_side") for v in final_b["members"].values())
missing = [n for n, (_, run, _) in MEMBERS.items() if not (run / "best.pt").exists()]
assert not missing, f"missing runs: {missing}"

# Every val frame (temporal context) and every test frame, with clip and frame number
val_meta = json.loads((DATA_ROOT / "val" / "annotations_coco.json").read_text())["images"]
VAL = [{"image_id": im["id"], "video": im["video"], "frame": im["frame_id"],
        "path": DATA_ROOT / "val" / im["file_name"], "shape": (im["height"], im["width"])} for im in val_meta]
test = pd.read_csv(TEST_CSV)
TEST = [{"image_id": k, "video": c, "frame": int(f), "path": DATA_ROOT / p, "shape": (h, w)}
        for k, c, f, p, w, h in zip(test["image_id"], test["clip"], test["frame"], test["file_name"], test["width"], test["height"])]
if FIXTURE:   # first 10 frames of one val clip (frames 1 and 9 are labelled) and the first 6 test frames
    VAL = [f for f in VAL if f["video"] == VAL[0]["video"] and f["frame"] <= 10]
    TEST = TEST[:6]
LABELLED = sorted({im["id"] for im in json.loads(GT_FILE.read_text())["images"]} & {f["image_id"] for f in VAL})
print(f"{len(MEMBERS)} members | val: {len(VAL)} frames ({len(LABELLED)} labelled) | test: {len(TEST)} frames")

11 members | val: 2846 frames (359 labelled) | test: 3811 frames


## 1. Predict every val frame with every member (GPU)

Each member's boxes are cached per split in `runs/ensemble/<split>_<member>.pkl` as `{image_id: (xyxy, score, class 0..7)}` in original pixels, at most 500 per frame. A finished cache is reused; an interrupted one resumes.

In [6]:
def load_member(name):
    family, run, _ = MEMBERS[name]
    if family == "yolo":
        return YOLO(str(run / "best.pt"))
    if family == "rtdetr":
        return RTDETR(str(run / "best.pt"))
    if family == "faster_rcnn":
        return load_faster_rcnn_run(run, DEVICE)
    return load_retinanet_run(run, DEVICE)


TV_ARGS = SimpleNamespace(scales=MS, hi_max_box=None, nms=0.5, slice=None, full_pass=False)   # infer.py's multi-scale settings


def predict_member(name, model, img_bgr):
    family, _, size = MEMBERS[name]
    if family in ("yolo", "rtdetr"):   # Person B's pipeline: full frame + flip, class-wise NMS
        passes = predict_passes(model, img_bgr, size, None, device=DEVICE, passes=("full", "flip"))
        return combine(passes, ("full", "flip"))
    rgb = torch.from_numpy(np.ascontiguousarray(img_bgr[:, :, ::-1])).permute(2, 0, 1)
    b, s, c = tv_predict_image(model, rgb, TV_ARGS, DEVICE)
    return b.numpy().astype(np.float32), s.numpy().astype(np.float32), c.numpy().astype(int) - 1


def cached_predictions(name, split, frames):
    path = OUT / f"{split}_{name}.pkl"
    cache = pickle.loads(path.read_bytes()) if path.exists() else {}
    todo = [f for f in frames if f["image_id"] not in cache]
    if todo:
        model = load_member(name)
        t0 = time.time()
        for n, f in enumerate(todo, 1):
            cache[f["image_id"]] = predict_member(name, model, cv2.imread(str(f["path"])))
            if n % 200 == 0 or n == len(todo):
                path.write_bytes(pickle.dumps(cache))
                print(f"  {split} {name}: {n}/{len(todo)} frames, {(time.time() - t0) / n:.2f} s/frame", flush=True)
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    return cache


val_preds = {}
for name in MEMBERS:
    print(name)
    val_preds[name] = cached_predictions(name, "valall", VAL)

yolo_low_scale_960
yolo_baseline_1280
yolo_final_960
yolo_final_1280
rtdetr_960
rtdetr_final_960
rtdetr_final_960_s025
frcnn_baseline_ms
retinanet_baseline_ms
yolo26m_1280_testres
  valall yolo26m_1280_testres: 200/2646 frames, 0.05 s/frame
  valall yolo26m_1280_testres: 400/2646 frames, 0.07 s/frame
  valall yolo26m_1280_testres: 600/2646 frames, 0.07 s/frame
  valall yolo26m_1280_testres: 800/2646 frames, 0.08 s/frame
  valall yolo26m_1280_testres: 1000/2646 frames, 0.07 s/frame
  valall yolo26m_1280_testres: 1200/2646 frames, 0.07 s/frame
  valall yolo26m_1280_testres: 1400/2646 frames, 0.08 s/frame
  valall yolo26m_1280_testres: 1600/2646 frames, 0.08 s/frame
  valall yolo26m_1280_testres: 1800/2646 frames, 0.09 s/frame
  valall yolo26m_1280_testres: 2000/2646 frames, 0.09 s/frame
  valall yolo26m_1280_testres: 2200/2646 frames, 0.09 s/frame
  valall yolo26m_1280_testres: 2400/2646 frames, 0.09 s/frame
  valall yolo26m_1280_testres: 2600/2646 frames, 0.09 s/frame
  valall yolo26m_1

## 2. Each member alone (labelled val frames)

In [ ]:
VAL_BY_ID = {f["image_id"]: f for f in VAL}
COLS = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR"]
results = []   # every scored configuration -> runs/ensemble/results.csv


def to_dets(boxes_by_id, ids):
    dets = []
    for i in ids:
        xyxy, s, c = boxes_by_id[i]
        dets += [{"image_id": i, "category_id": int(k) + 1, "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                  "score": float(v)} for (x1, y1, x2, y2), v, k in zip(xyxy, s, c)]
    return dets


def score(boxes_by_id, label, **info):
    m = evaluate_detections(GT_FILE, to_dets(boxes_by_id, LABELLED), image_ids=LABELLED)
    results.append({"config": label, **info, **m})
    return m


single = pd.DataFrame({n: score(val_preds[n], n, kind="single") for n in MEMBERS}).T[COLS]
single.sort_values("mAP", ascending=False).round(4)

,mAP,AP50,AP75,AP_small,AP_medium,AP_large,AR
rtdetr_final_960_s025,0.2534,0.4891,0.2310,0.1089,0.2162,0.4150,0.4368
rtdetr_final_960,0.2486,0.4774,0.2260,0.1103,0.2182,0.4243,0.4050
rtdetr_960,0.2341,0.4480,0.2173,0.0912,0.1975,0.4114,0.3899
yolo_low_scale_960,0.2206,0.4145,0.2067,0.1077,0.1772,0.4036,0.3562
yolo_baseline_1280,0.2204,0.4343,0.1994,0.1527,0.2145,0.3479,0.3683
frcnn_baseline_ms,0.2192,0.4622,0.1809,0.1101,0.2017,0.4057,0.4586
yolo_final_1280,0.1984,0.3988,0.1747,0.1097,0.1961,0.3274,0.3271
yolo_final_960,0.1951,0.3778,0.1814,0.1056,0.1748,0.3305,0.3236
retinanet_baseline_ms,0.1913,0.3753,0.1724,0.0921,0.1793,0.3535,0.3983


The YOLO / RT-DETR rows should match the `+flip` rows of `09_final_pipeline.ipynb`.

## 3. Versions A and B: weighted box fusion

Each configuration fuses its members' boxes frame by frame with Person B's `fuse` (WBF, equal weights). Only the labelled frames are fused here, so the grid is quick.

In [ ]:
def fuse_frames(names, iou, ids):
    return {i: fuse([val_preds[n][i] for n in names], VAL_BY_ID[i]["shape"], iou=iou) for i in ids}


OURS = ["frcnn_baseline_ms", "retinanet_baseline_ms"]
SETS = {
    "A: B's 7": B_MEMBERS,
    "B: B's 7 + Faster R-CNN": B_MEMBERS + OURS[:1],
    "B: B's 7 + RetinaNet": B_MEMBERS + OURS[1:],
    "B: B's 7 + Faster R-CNN + RetinaNet": B_MEMBERS + OURS,
}
WBF_IOUS = [0.6, 0.65, 0.7, 0.75]

fused_rows = {}
for label, names in SETS.items():
    for iou in WBF_IOUS:
        t0 = time.time()
        m = score(fuse_frames(names, iou, LABELLED), label, kind="fused", wbf_iou=iou)
        fused_rows[(label, iou)] = m
        print(f"{label}, WBF IoU {iou}: mAP {m['mAP']:.4f}  ({time.time() - t0:.0f} s)")
fused_table = pd.DataFrame(fused_rows).T[COLS]
fused_table.round(4)

A: B's 7, WBF IoU 0.6: mAP 0.2954  (45 s)
A: B's 7, WBF IoU 0.65: mAP 0.2977  (44 s)
A: B's 7, WBF IoU 0.7: mAP 0.2968  (45 s)
A: B's 7, WBF IoU 0.75: mAP 0.2948  (46 s)
B: B's 7 + Faster R-CNN, WBF IoU 0.6: mAP 0.2931  (48 s)
B: B's 7 + Faster R-CNN, WBF IoU 0.65: mAP 0.2946  (47 s)
B: B's 7 + Faster R-CNN, WBF IoU 0.7: mAP 0.2930  (48 s)
B: B's 7 + Faster R-CNN, WBF IoU 0.75: mAP 0.2907  (48 s)
B: B's 7 + RetinaNet, WBF IoU 0.6: mAP 0.2983  (48 s)
B: B's 7 + RetinaNet, WBF IoU 0.65: mAP 0.2998  (48 s)
B: B's 7 + RetinaNet, WBF IoU 0.7: mAP 0.2987  (48 s)
B: B's 7 + RetinaNet, WBF IoU 0.75: mAP 0.2964  (49 s)
B: B's 7 + Faster R-CNN + RetinaNet, WBF IoU 0.6: mAP 0.2942  (52 s)
B: B's 7 + Faster R-CNN + RetinaNet, WBF IoU 0.65: mAP 0.2952  (52 s)
B: B's 7 + Faster R-CNN + RetinaNet, WBF IoU 0.7: mAP 0.2932  (52 s)
B: B's 7 + Faster R-CNN + RetinaNet, WBF IoU 0.75: mAP 0.2917  (52 s)


mAP    AP50    AP75  AP_small  \
A: B's 7                            0.60  0.2954  0.5505  0.2829    0.1450   
                                    0.65  0.2977  0.5502  0.2879    0.1455   
                                    0.70  0.2968  0.5451  0.2884    0.1448   
                                    0.75  0.2948  0.5370  0.2880    0.1449   
B: B's 7 + Faster R-CNN             0.60  0.2931  0.5514  0.2788    0.1463   
                                    0.65  0.2946  0.5490  0.2821    0.1472   
                                    0.70  0.2930  0.5425  0.2813    0.1456   
                                    0.75  0.2907  0.5316  0.2836    0.1418   
B: B's 7 + RetinaNet                0.60  0.2983  0.5511  0.2887    0.1439   
                                    0.65  0.2998  0.5494  0.2914    0.1435   
                                    0.70  0.2987  0.5435  0.2915    0.1426   
                                    0.75  0.2964  0.5341  0.2909    0.1387   
B: B's 7 + Faster R-CNN + RetinaNet 0.60  0.2942  0.5497  0.2797    0.1444   
                                    0.65  0.2952  0.5477  0.2848    0.1446   
                                    0.70  0.2932  0.5408  0.2839    0.1421   
                                    0.75  0.2917  0.5298  0.2859    0.1355   

                                          AP_medium  AP_large      AR  
A: B's 7                            0.60     0.2507    0.4873  0.4852  
                                    0.65     0.2535    0.4897  0.4901  
                                    0.70     0.2535    0.4835  0.4942  
                                    0.75     0.2517    0.4806  0.4980  
B: B's 7 + Faster R-CNN             0.60     0.2530    0.4950  0.4904  
                                    0.65     0.2551    0.4940  0.4930  
                                    0.70     0.2539    0.4901  0.4978  
                                    0.75     0.2521    0.4855  0.5040  
B: B's 7 + RetinaNet                0.60     0.2561    0.4979  0.5160  
                                    0.65     0.2582    0.4972  0.5208  
                                    0.70     0.2581    0.4919  0.5257  
                                    0.75     0.2562    0.4892  0.5297  
B: B's 7 + Faster R-CNN + RetinaNet 0.60     0.2568    0.4987  0.5173  
                                    0.65     0.2586    0.4966  0.5227  
                                    0.70     0.2571    0.4925  0.5274  
                                    0.75     0.2558    0.4895  0.5344

In [ ]:
A_KEY = ("A: B's 7", B_WBF_IOU)
BEST_KEY = max(fused_rows, key=lambda k: fused_rows[k]["mAP"])
BEST_NAMES, BEST_IOU = SETS[BEST_KEY[0]], BEST_KEY[1]
print(f"A (Person B's config): mAP {fused_rows[A_KEY]['mAP']:.4f}  (09 reported 0.2977)")
print(f"best: {BEST_KEY[0]}, WBF IoU {BEST_IOU}: mAP {fused_rows[BEST_KEY]['mAP']:.4f}")

A (Person B's config): mAP 0.2977  (09 reported 0.2977)
best: B: B's 7 + RetinaNet, WBF IoU 0.65: mAP 0.2998


**Observations:** _fill in_

## 4. Version C: temporal rescoring of the best fusion

The best configuration is fused on **every** val frame, then `temporal.py`'s linking and rescoring run on the fused boxes with the same grid as `temporal.py` (tuned on val, scored on the labelled frames). Fusing all 2846 frames takes a while on the CPU.

In [ ]:
def with_meta(boxes_by_id, frames):
    # Flat detection list with clip / frame number, as temporal.py expects
    dets = []
    for f in frames:
        xyxy, s, c = boxes_by_id[f["image_id"]]
        dets += [{"image_id": f["image_id"], "video": f["video"], "frame": f["frame"], "category_id": int(k) + 1,
                  "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(v)}
                 for (x1, y1, x2, y2), v, k in zip(xyxy, s, c)]
    return dets


t0 = time.time()
best_val = fuse_frames(BEST_NAMES, BEST_IOU, [f["image_id"] for f in VAL])
val_dets = with_meta(best_val, VAL)
print(f"fused {len(VAL)} val frames in {(time.time() - t0) / 60:.1f} min: {len(val_dets)} boxes")
labelled = set(LABELLED)


def score_dets(dets, label, **info):
    m = evaluate_detections(GT_FILE, [d for d in dets if d["image_id"] in labelled], image_ids=LABELLED)
    results.append({"config": label, **info, **m})
    return m


GRID = dict(link_iou=[0.3, 0.5], link_min=[0.05], lam=[0.3, 0.5, 0.7], top_k=[5, 15], min_len=[3], single_penalty=[1.0, 0.7])
temporal_rows = {}
tracks = {}
for link_iou in GRID["link_iou"]:
    tracks[link_iou] = link_tracks(val_dets, link_iou, GRID["link_min"][0])
    for lam in GRID["lam"]:
        for top_k in GRID["top_k"]:
            for pen in GRID["single_penalty"]:
                new = rescore(val_dets, tracks[link_iou], lam, top_k, GRID["min_len"][0], pen)
                cfg = dict(link_iou=link_iou, lam=lam, top_k=top_k, single_penalty=pen)
                temporal_rows[tuple(cfg.values())] = score_dets(
                    [{**d, "score": float(s)} for d, s in zip(val_dets, new)], "C: best + temporal", kind="temporal", **cfg)
temporal_table = pd.DataFrame(temporal_rows).T[COLS]
temporal_table.index.names = ["link_iou", "lam", "top_k", "single_penalty"]
BEST_T = dict(zip(temporal_table.index.names, temporal_table["mAP"].idxmax()))
print("best temporal setting:", BEST_T)
temporal_table.sort_values("mAP", ascending=False).head(8).round(4)

fused 2846 val frames in 2.8 min: 1423000 boxes
best temporal setting: {'link_iou': np.float64(0.5), 'lam': np.float64(0.5), 'top_k': np.int64(15), 'single_penalty': np.float64(1.0)}


mAP    AP50    AP75  AP_small  \
link_iou lam top_k single_penalty                                     
0.5      0.5 15    1.0             0.3051  0.5588  0.2949    0.1790   
0.3      0.5 15    1.0             0.3049  0.5592  0.2946    0.1858   
         0.3 15    1.0             0.3048  0.5584  0.2953    0.1606   
0.5      0.3 15    1.0             0.3047  0.5578  0.2954    0.1557   
0.3      0.3 5     1.0             0.3044  0.5576  0.2951    0.1609   
0.5      0.3 5     1.0             0.3042  0.5569  0.2949    0.1559   
0.3      0.5 15    0.7             0.3041  0.5580  0.2936    0.1853   
         0.3 15    0.7             0.3041  0.5573  0.2944    0.1602   

                                   AP_medium  AP_large      AR  
link_iou lam top_k single_penalty                               
0.5      0.5 15    1.0                0.2636    0.5034  0.5208  
0.3      0.5 15    1.0                0.2630    0.5046  0.5208  
         0.3 15    1.0                0.2620    0.5048  0.5208  
0.5      0.3 15    1.0                0.2622    0.5040  0.5208  
0.3      0.3 5     1.0                0.2618    0.5035  0.5208  
0.5      0.3 5     1.0                0.2617    0.5031  0.5208  
0.3      0.5 15    0.7                0.2628    0.5042  0.5208  
         0.3 15    0.7                0.2618    0.5044  0.5208

## Summary on val

In [ ]:
c_best = temporal_rows[tuple(BEST_T.values())]
summary = pd.DataFrame({
    f"A: Person B's ensemble (IoU {B_WBF_IOU})": fused_rows[A_KEY],
    f"{BEST_KEY[0]} (IoU {BEST_IOU})": fused_rows[BEST_KEY],
    "C: best + temporal": c_best,
}).T[COLS]
summary["delta mAP vs A"] = summary["mAP"] - summary["mAP"].iloc[0]
pd.DataFrame(results).to_csv(OUT / "results.csv", index=False)
summary.round(4)

,mAP,AP50,AP75,AP_small,AP_medium,AP_large,AR,delta mAP vs A
A: Person B's ensemble (IoU 0.65),0.2977,0.5502,0.2879,0.1455,0.2535,0.4897,0.4901,0.0000
B: B's 7 + RetinaNet (IoU 0.65),0.2998,0.5494,0.2914,0.1435,0.2582,0.4972,0.5208,0.0021
C: best + temporal,0.3051,0.5588,0.2949,0.1790,0.2636,0.5034,0.5208,0.0073


**Observations:** _fill in_

## 5. Test submissions

Every member predicts the test frames (GPU, cached like val; all members, so `ensemble_weights.ipynb` can combine any of them), then three submissions are written to `runs/ensemble/submissions/`:
`A_b_ensemble.csv`, `B_best_ensemble.csv` (skipped if the best configuration is A itself) and `C_best_temporal.csv`.

In [7]:
# Resuming after a kernel restart: run Setup, "Members and frames" and section 1 (it only loads the cached val
# predictions, a few seconds), then this cell. The chosen configuration is read back from runs/ensemble/results.csv.
if "BEST_NAMES" not in globals():
    res = pd.read_csv(OUT / "results.csv")
    OURS = ["frcnn_baseline_ms", "retinanet_baseline_ms"]
    SETS = {"A: B's 7": B_MEMBERS, "B: B's 7 + Faster R-CNN": B_MEMBERS + OURS[:1],
            "B: B's 7 + RetinaNet": B_MEMBERS + OURS[1:], "B: B's 7 + Faster R-CNN + RetinaNet": B_MEMBERS + OURS}
    f = res[res["kind"] == "fused"].sort_values("mAP", ascending=False).iloc[0]
    A_KEY, BEST_KEY = ("A: B's 7", B_WBF_IOU), (f["config"], float(f["wbf_iou"]))
    BEST_NAMES, BEST_IOU = SETS[f["config"]], float(f["wbf_iou"])
    t = res[res["kind"] == "temporal"].sort_values("mAP", ascending=False).iloc[0]
    BEST_T = dict(link_iou=float(t["link_iou"]), lam=float(t["lam"]), top_k=int(t["top_k"]), single_penalty=float(t["single_penalty"]))
    GRID = dict(link_min=[0.05], min_len=[3])

    def with_meta(boxes_by_id, frames):
        dets = []
        for fr in frames:
            xyxy, s, c = boxes_by_id[fr["image_id"]]
            dets += [{"image_id": fr["image_id"], "video": fr["video"], "frame": fr["frame"], "category_id": int(k) + 1,
                      "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(v)}
                     for (x1, y1, x2, y2), v, k in zip(xyxy, s, c)]
        return dets
    print(f"resumed: {BEST_KEY}, temporal {BEST_T}")
for name in ("val_dets", "best_val", "tracks"):   # free the val fusion from section 4 before the test set
    globals().pop(name, None)

test_preds = {}
for name in MEMBERS:   # every member, so ensemble_weights.ipynb can try any combination on test
    print(name)
    test_preds[name] = cached_predictions(name, "test", TEST)


def fuse_test(names, iou):
    t0 = time.time()
    out = {f["image_id"]: fuse([test_preds[n][f["image_id"]] for n in names], f["shape"], iou=iou) for f in TEST}
    print(f"fused {len(TEST)} test frames in {(time.time() - t0) / 60:.1f} min")
    return out


def save(boxes_by_id, name):
    sub = write_submission({k: to_submission_rows(v) for k, v in boxes_by_id.items()}, TEST_CSV, OUT / "submissions" / name)
    n_boxes = sub.PredictionString.fillna("").str.split().str.len().fillna(0) // 6
    assert n_boxes.max() <= 500
    print(f"wrote {name}: {len(sub)} rows, {int((n_boxes > 0).sum())} with detections")


best_test = fuse_test(BEST_NAMES, BEST_IOU)
if BEST_KEY == A_KEY:
    save(best_test, "A_b_ensemble.csv")
else:
    save(fuse_test(B_MEMBERS, B_WBF_IOU), "A_b_ensemble.csv")
    save(best_test, "B_best_ensemble.csv")

test_dets = with_meta(best_test, TEST)
t = link_tracks(test_dets, BEST_T["link_iou"], GRID["link_min"][0])
new = rescore(test_dets, t, BEST_T["lam"], int(BEST_T["top_k"]), GRID["min_len"][0], BEST_T["single_penalty"])
rows = {}
for d, s in zip(test_dets, new):
    x, y, w, h = d["bbox"]
    rows.setdefault(d["image_id"], []).append((d["category_id"] - 1, float(s), x, y, w, h))
sub = write_submission(rows, TEST_CSV, OUT / "submissions" / "C_best_temporal.csv")
print(f"wrote C_best_temporal.csv: {len(sub)} rows")

resumed: ("B: B's 7 + RetinaNet", 0.65), temporal {'link_iou': 0.5, 'lam': 0.5, 'top_k': 15, 'single_penalty': 1.0}
yolo_low_scale_960
yolo_baseline_1280
yolo_final_960
yolo_final_1280
rtdetr_960
rtdetr_final_960
rtdetr_final_960_s025
frcnn_baseline_ms
  test frcnn_baseline_ms: 200/3811 frames, 0.11 s/frame
  test frcnn_baseline_ms: 400/3811 frames, 0.10 s/frame
  test frcnn_baseline_ms: 600/3811 frames, 0.11 s/frame
  test frcnn_baseline_ms: 800/3811 frames, 0.10 s/frame
  test frcnn_baseline_ms: 1000/3811 frames, 0.10 s/frame
  test frcnn_baseline_ms: 1200/3811 frames, 0.10 s/frame
  test frcnn_baseline_ms: 1400/3811 frames, 0.10 s/frame
  test frcnn_baseline_ms: 1600/3811 frames, 0.10 s/frame
  test frcnn_baseline_ms: 1800/3811 frames, 0.10 s/frame
  test frcnn_baseline_ms: 2000/3811 frames, 0.10 s/frame
  test frcnn_baseline_ms: 2200/3811 frames, 0.10 s/frame
  test frcnn_baseline_ms: 2400/3811 frames, 0.10 s/frame
  test frcnn_baseline_ms: 2600/3811 frames, 0.10 s/frame
  test frc